## Setup

All the paths used by this exercise are set in the cell below.

In [ ]:
#############################################################
# ALL PATHS ARE SET HERE
# If the data or the software moves, this is the ONLY cell you
# need to change. No cell below this one uses a full path.
#############################################################

# where the shared data lives
DATA=/course/data/PCA/human_lowdepth

# where you will do the exercise
WORK_DIR=$HOME/pca_low_depth_human

mkdir -p $WORK_DIR
echo $WORK_DIR > $HOME/.pca_low_depth_human_workdir
cd $WORK_DIR

# link the input files into the working folder
cp -sf $DATA/1000G5pops.inputgl.beagle.gz .
cp -sf $DATA/1000G5pops.pop.info .
cp -sf $DATA/1000G5popsAdmixK3seed3.qopt .
cp -sf $DATA/1000G5popsAdmixK4seed9.qopt .

echo --programs that are installed:--
which pcangsd

echo; echo --- files in folder ---
ls

In [ ]:
# the working directory was set in the first cell of the notebook
work_d <- readLines(path.expand("~/.pca_low_depth_human_workdir"))[1]
setwd(work_d)
getwd()

# PCA for low depth sequencing using PCAngsd

In this exercise we will use PCAngsd to perform a PCA from **genotype
likelihoods**, without ever calling a genotype. The data are five populations
from the 1000 Genomes project, sequenced at low depth.

If you have not done the [MDS and PCA by hand](pca_mds_and_svd.ipynb) exercise,
do that one first — it builds the method this notebook now runs with a tool.

Copy data to your newly created folder

In [ ]:
# the files were linked into the folder in the setup cell
ls *.qopt *.pop.info *.beagle.gz


Look inside the first lines in the population information file

In [ ]:
head 1000G5pops.pop.info


See the number of individuals for each population from the sample file

In [ ]:
# summarise the first column
cut -f1 1000G5pops.pop.info |  uniq -c

**Question**
 - How many individuals are there in each population, and how many populations in total?

Count the number of lines in the genotype likelihood file

In [ ]:
zcat 1000G5pops.inputgl.beagle.gz | wc -l 

**Question**
 - A beagle genotype likelihood file has one line per site plus a header, and three columns per individual. From the line count, how many sites are there?

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/pca/quiz/pcangsd.json")


 
 ## Run PCANGSD to perform PCA
 First let's get a list of the options in PCAngsd


In [ ]:
pcangsd -h

Run PCANGSD on your genotype likelihood data using 5 CPU threads (will take ~1min)

In [ ]:
pcangsd -b 1000G5pops.inputgl.beagle.gz -o PCANGSD1000G -t 5

**Question**
 - PCAngsd estimated a covariance matrix rather than calling genotypes. Why is that the right thing to do at low depth?

The program estimates the covariance matrix that can then be used for PCA. Look at the output from the program

 - How many significant PCA was used by PCAngsd (see MAP test in output)?

Plot the results in R

In [ ]:
# Read covariance matrix estimated by PCAngsd
C <- as.matrix(read.table("PCANGSD1000G.cov"))

# Read population labels for each individuals
pop<-read.table("1000G5pops.pop.info",stringsAsFactors=T)

# Estimate the eigenvectors (principal components) from the covariance matrix
e <- eigen(C)
plot(e$vectors[,c(1,2)],col=pop[,1],xlab="PC1",ylab="PC2")
legend("left",fill=1:5,levels(pop[,1]))


# Estimate the eigenvectors (principal components) from the covariance matrix
e <- eigen(C)
plot(e$vectors[,c(1,3)],col=pop[,1],xlab="PC1",ylab="PC3")
legend("left",fill=1:5,levels(pop[,1]))

**Questions**
 - Which populations sit close together, and which are far apart?
 - Does PC1 separate the same populations as PC2?

Compare with the estimate admixture proportions (a NGSadmix analysis)



In [ ]:
source("https://raw.githubusercontent.com/GenisGE/evalAdmix/refs/heads/master/visFuns.R")

par(mfrow=2:1)
## read and plot the output from NGSadmix from the Tuesday's exercises
pop<-read.table("1000G5pops.pop.info",as.is=T)
q<-read.table("1000G5popsAdmixK3seed3.qopt")
# sort indiivduals by population and within populaoitn by admixture proportion
ord<-orderInds(pop = pop[,1], q=q) 

#plot
barplot(t(q)[,ord],col=2:10,space=0,border=NA,xlab="Individuals",
        ylab="Admixture proportions",main="K=3")
text(sort(tapply(1:nrow(pop),pop[ord,1],mean)),-0.05,
     unique(pop[ord,1]),xpd=T) # add population labels
abline(v=cumsum(sapply(unique(pop[ord,1]),
                       function(x){sum(pop[ord,1]==x)})),col=1,lwd=1.2)

## read for K=4
pop<-read.table("1000G5pops.pop.info",as.is=T)
q<-read.table("1000G5popsAdmixK4seed9.qopt")
plot
barplot(t(q)[,ord],col=2:10,space=0,border=NA,xlab="Individuals",
        ylab="Admixture proportions",main="K=4")
text(sort(tapply(1:nrow(pop),pop[ord,1],mean)),-0.05,unique(pop[ord,1]),
     xpd=T) # add population labels
abline(v=cumsum(sapply(unique(pop[ord,1]),
                       function(x){sum(pop[ord,1]==x)})),col=1,lwd=1.2)

**Questions**
 - In the PCA plot can you identify the Mexicans with only European ancestry?
 - What about the ones with mostly Native American ancestry?
 - What does the PCA tell you that the admixture proportions do not?

### Run the cell below to take the quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/pca/quiz/pcangsd.json")

 - In the PCA plot can you identify the Mexicans with only European ancestry?
 - What about the African American with East Asian ancestry?
 - Based on the PCA would you have reached the same conclusion as the admixture proportions?

## What if we dont use many iterations in PCAngnd

Try the same analysis but with only one iteration in the algorithm so that the result has not converged. Then we do not fully take the population structure into account when filling in the missing information


In [ ]:

pcangsd -b 1000G5pops.inputgl.beagle.gz -o PCANGSD1000G_iter0 -t 5 --iter 1 -e 1

**Question**
 - This run used `--iter 1 -e 1`, so the individual allele frequencies are barely estimated. What do you expect to change?

wait for the analysis to finish and then plot the results in R using the code below

In [ ]:
# Read covariance matrix estimated by PCAngsd
C <- as.matrix(read.table("PCANGSD1000G_iter0.cov"))

# Read population labels for each individuals
pop<-read.table("1000G5pops.pop.info",stringsAsFactors=T)

# Estimate the eigenvectors (Principal components) from the covariance matrix
e <- eigen(C)
plot(e$vectors[,1:2],col=pop[,1],xlab="PC1",ylab="PC2")
legend("top",fill=1:5,levels(pop[,1]))


**Questions**
 - Do you see any difference from the first plot?
 - Would any of your conclusions change?

 - Do you see any difference?
 - Would any of your conclusions change? (compared to the previous PCA plot)

## Converting a PCA into admixture proportions
Let's try to use the PCA to infer admixture proportions based on the first 2 principal components. For the optimization we will use a small penalty on the admixture proportions (alpha). This is a way to convert your PCA into admixture proportions:


In [ ]:

pcangsd -b 1000G5pops.inputgl.beagle.gz -o PCANGSD1000G -t 5 --admix --admix-alpha 50 


**Question**
 - `--admix` makes PCAngsd estimate admixture proportions from the PCA itself. How does that differ from running NGSadmix?


Plot the results in R



In [ ]:
# Read the admixture proportions estimated from the PCA
q<-read.table("PCANGSD1000G.admix.4.Q")

# Read population labels for each individuals
pop<-read.table("1000G5pops.pop.info",stringsAsFactors=T)

## Order according to population
ord<-orderInds(pop = pop[,1], q=q) # sort indiivduals by population and within populaoitn by admixture proportion

#plot
barplot(t(q)[,ord],col=2:10,space=0,border=NA,xlab="Individuals",
        ylab="Admixture proportions")
text(sort(tapply(1:nrow(pop),pop[ord,1],mean)),-0.05,
     unique(pop[ord,1]),xpd=T) # add population labels
abline(v=cumsum(sapply(unique(pop[ord,1]),
                       function(x){sum(pop[ord,1]==x)})),col=1,lwd=1.2)

**Question**
 - How does this compare with the NGSadmix result you plotted earlier?


 - how does this compare to the results from an admixture proportion analysis (the NGSadmix analysis above)?
